In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import time
from functools import reduce

import parreg.process_config as pc
from parreg import funcs_clust, funcs_dist
from parreg.logging_config import setup_logging
setup_logging()

import logging
logger = logging.getLogger(__name__)

In [ ]:
# read and validate config
config_file = Path('/home/yuqiong.liu/work/Gitlab/ngen-regionalization/configs/config.yaml')
if not config_file.exists():
    raise FileNotFoundError(config_file)

config = pc.load_and_validate_config(config_file)

In [ ]:
# process by VPU
vpu = config.general.vpu_list[0]

In [ ]:
# get receivers and qualified donors in the VPU, and compute pairwise spatial distances between them
donors, receivers, df_dist_spatial = pc.get_donors_receivers(config, vpu)

In [ ]:
# assemble the attribute data for donors and receivers
donors, receivers, df_attrs_all = pc.process_attr_data(config, vpu, donors, receivers, df_dist_spatial)

In [ ]:
# detemine whether the catchments are snowy (as snowy and non-snowy catchments are processed separately)
df_attrs_all = pc.set_snow_flag(df_attrs_all, config.algorithms.general.min_snow_frac)

In [ ]:
# loop through regionalization algorithms to generate donor-receiver pairings
pc.generate_pairing(config, vpu, df_attrs_all, df_dist_spatial)